# PhoBERT fine-tune — OOF cho stacking

**Mục đích:** fine-tune `vinai/phobert-base` trên `comment_seg`, fusion 52 đặc trưng dense, sinh xác suất **OOF (3 fold)**, dev và test để đưa vào stacking.

**Yêu cầu môi trường:**

```powershell
python -m pip uninstall -y torch torchvision torchaudio
python -m pip install torch --index-url https://download.pytorch.org/whl/cu126
python -m pip install transformers sentencepiece datasets accelerate
```

Kiểm tra: `torch.cuda.is_available()` phải là `True`.

**Đầu vào:**
- `artifacts/phobert/{train,dev,test}.parquet` (từ `phobert_preprocess.ipynb`)
- `artifacts/dense/dense_*.npy`
- `artifacts/labels/y_*.npy`

**Đầu ra:**
- `artifacts/probs/oof_phobert/phobert_oof.npy` (14,306 × 4)
- `phobert_dev.npy` (1,590 × 4)
- `phobert_test.npy` (3,974 × 4)

**Cấu hình cho RTX 2050 4 GB:** `max_len=128`, batch 8, AMP fp16, 3 epoch, 3 fold. Nếu OOM: giảm batch còn 4, bật `GRAD_CHECKPOINT=True`.

**Thời gian ước tính:** ~45–60 phút/fold → **2.5–3 giờ** cho 3 fold.

In [1]:
# ============================================================
# 1. KIỂM TRA MÔI TRƯỜNG
# ============================================================
import math
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")

try:
    import torch
    import torch.nn as nn
    from torch.utils.data import DataLoader, Dataset
    from torch.optim import AdamW
    from transformers import AutoModel, AutoTokenizer, get_linear_schedule_with_warmup
except ImportError as exc:
    raise ImportError(
        "Thiếu torch/transformers. Cài theo lệnh trong markdown đầu notebook:\n"
        "  python -m pip uninstall -y torch torchvision torchaudio\n"
        "  python -m pip install torch --index-url https://download.pytorch.org/whl/cu126\n"
        "  python -m pip install transformers sentencepiece datasets accelerate"
    ) from exc

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
USE_AMP = DEVICE == "cuda"
print("torch:", torch.__version__, "| CUDA build:", torch.version.cuda)
print("device:", DEVICE, "| AMP:", USE_AMP)
if DEVICE == "cpu":
    print("CẢNH BÁO: không có CUDA — fine-tune trên CPU sẽ rất chậm.")


def find_project_root(start=None, marker="Data/ds102_train.csv"):
    p = Path(start).resolve() if start is not None else Path.cwd().resolve()
    for cand in [p, *p.parents]:
        if (cand / marker).exists():
            return cand
    raise FileNotFoundError(f"Không tìm thấy thư mục gốc chứa '{marker}'")


PROJECT_ROOT = find_project_root()
ARTIFACT_DIR = PROJECT_ROOT / "artifacts"
PHOBERT_TXT_DIR = ARTIFACT_DIR / "phobert"
OOF_DIR = ARTIFACT_DIR / "probs" / "oof_phobert"
OOF_DIR.mkdir(parents=True, exist_ok=True)
print("OOF_DIR:", OOF_DIR)

torch: 2.14.1+cu126 | CUDA build: 12.6
device: cuda | AMP: True
OOF_DIR: D:\DS102_NguyenKhacHuy_24520679\Final project\artifacts\probs\oof_phobert


In [2]:
# ============================================================
# 2. CẤU HÌNH & DỮ LIỆU
# ============================================================
MODEL_NAME = "vinai/phobert-base"
MAX_LEN = 128
BATCH_SIZE = 8
EPOCHS = 3
LR = 2e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.1
N_FOLDS = 3
SEED = 42
CLASS_WEIGHT_MODE = "sqrt"   # sqrt | balanced | none
GRAD_CHECKPOINT = True
PATIENCE = 1
NUM_WORKERS = 0

np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# --- text đã tách từ ---
train_df = pd.read_parquet(PHOBERT_TXT_DIR / "train.parquet")
dev_df = pd.read_parquet(PHOBERT_TXT_DIR / "dev.parquet")
test_df = pd.read_parquet(PHOBERT_TXT_DIR / "test.parquet")

texts_train = train_df["comment_seg"].fillna("").tolist()
texts_dev = dev_df["comment_seg"].fillna("").tolist()
texts_test = test_df["comment_seg"].fillna("").tolist()

y_train = np.load(ARTIFACT_DIR / "labels" / "y_train.npy")
y_dev = np.load(ARTIFACT_DIR / "labels" / "y_dev.npy")

dense_train = np.load(ARTIFACT_DIR / "dense" / "dense_train.npy").astype(np.float32)
dense_dev = np.load(ARTIFACT_DIR / "dense" / "dense_dev.npy").astype(np.float32)
dense_test = np.load(ARTIFACT_DIR / "dense" / "dense_test.npy").astype(np.float32)

assert len(texts_train) == len(y_train) == len(dense_train)
assert len(texts_dev) == len(y_dev) == len(dense_dev)
assert len(texts_test) == len(dense_test)

print("train:", len(texts_train), "| dev:", len(texts_dev), "| test:", len(texts_test))
print("dense dims:", dense_train.shape[1])
print("nhãn train:", dict(zip(*np.unique(y_train, return_counts=True))))

train: 14306 | dev: 1590 | test: 3974
dense dims: 52
nhãn train: {np.int64(0): np.int64(10512), np.int64(1): np.int64(211), np.int64(2): np.int64(1103), np.int64(3): np.int64(2480)}


## 3. Tokenizer & Dataset

Tokenize một lần cho cả 3 split (nhanh hơn tokenize trong `__getitem__`).

In [3]:
# ============================================================
# 3. TOKENIZER & DATASET
# ============================================================
try:
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
    print("Dùng fast tokenizer.")
except Exception as exc:
    print("Fast tokenizer lỗi:", exc)
    print("Thử slow tokenizer (có thể cần: python -m pip install fastBPE).")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=False)


def encode(texts):
    return tokenizer(
        texts, truncation=True, max_length=MAX_LEN,
        padding="max_length", return_tensors="pt",
    )


print("Đang tokenize...")
enc_train = encode(texts_train)
enc_dev = encode(texts_dev)
enc_test = encode(texts_test)
print("enc_train:", enc_train["input_ids"].shape)


def subset_enc(enc, idx):
    idx = torch.as_tensor(idx, dtype=torch.long)
    return {
        "input_ids": enc["input_ids"][idx],
        "attention_mask": enc["attention_mask"][idx],
    }


class ReviewDataset(Dataset):
    def __init__(self, enc, dense, labels=None):
        self.input_ids = enc["input_ids"]
        self.attention_mask = enc["attention_mask"]
        self.dense = torch.tensor(dense, dtype=torch.float32)
        self.labels = None if labels is None else torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return self.input_ids.shape[0]

    def __getitem__(self, i):
        item = {
            "input_ids": self.input_ids[i],
            "attention_mask": self.attention_mask[i],
            "dense": self.dense[i],
        }
        if self.labels is not None:
            item["labels"] = self.labels[i]
        return item


print("Đã tạo Dataset class.")

Dùng fast tokenizer.
Đang tokenize...
enc_train: torch.Size([14306, 128])
Đã tạo Dataset class.


## 4. Model: PhoBERT + fusion metadata

```
comment_seg → PhoBERT → [CLS] (768)
52 dense → Linear(52→64) → ReLU ─┐
                                 ├→ concat → Dropout → Linear(832→4)
```

In [4]:
# ============================================================
# 4. MODEL
# ============================================================
class PhoBERTMeta(nn.Module):
    def __init__(self, model_name, n_dense, n_classes=4, dropout=0.2):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        hidden = self.encoder.config.hidden_size
        self.dense_proj = nn.Sequential(
            nn.Linear(n_dense, 64), nn.ReLU(), nn.Dropout(dropout)
        )
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden + 64, n_classes)

    def forward(self, input_ids, attention_mask, dense):
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        cls = out.last_hidden_state[:, 0]
        d = self.dense_proj(dense)
        return self.classifier(self.dropout(torch.cat([cls, d], dim=1)))


def class_weights(y, mode):
    counts = np.bincount(y, minlength=4).astype(float)
    balanced = len(y) / (4.0 * np.maximum(counts, 1.0))
    if mode == "balanced":
        vals = balanced
    elif mode == "sqrt":
        vals = np.sqrt(balanced)
    else:
        vals = np.ones(4)
    return torch.tensor(vals, dtype=torch.float32)


print("Đã định nghĩa model. Weight sqrt:", class_weights(y_train, "sqrt").numpy().round(2).tolist())

Đã định nghĩa model. Weight sqrt: [0.5799999833106995, 4.119999885559082, 1.7999999523162842, 1.2000000476837158]


## 5. Hàm train một fold

- Loss: CrossEntropy có class weight (mặc định `sqrt`).
- AMP fp16, gradient clipping 1.0.
- Early stopping theo **macro-F1 trên dev**.
- Trả về xác suất cho fold validation, dev, test.

In [5]:
# ============================================================
# 5. TRAIN MỘT FOLD
# ============================================================
@torch.no_grad()
def predict(model, loader):
    model.eval()
    probs = []
    for batch in loader:
        input_ids = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        dense = batch["dense"].to(DEVICE)
        with torch.autocast(device_type=DEVICE, enabled=USE_AMP):
            logits = model(input_ids, attention_mask, dense)
        probs.append(torch.softmax(logits.float(), dim=1).cpu().numpy())
    return np.vstack(probs)


def run_fold(fold, tr_idx, va_idx):
    t0 = time.time()

    scaler = StandardScaler().fit(dense_train[tr_idx])
    d_tr = scaler.transform(dense_train[tr_idx]).astype(np.float32)
    d_va = scaler.transform(dense_train[va_idx]).astype(np.float32)
    d_dv = scaler.transform(dense_dev).astype(np.float32)
    d_te = scaler.transform(dense_test).astype(np.float32)

    train_ds = ReviewDataset(subset_enc(enc_train, tr_idx), d_tr, y_train[tr_idx])
    va_ds = ReviewDataset(subset_enc(enc_train, va_idx), d_va)
    dev_ds = ReviewDataset(enc_dev, d_dv)
    test_ds = ReviewDataset(enc_test, d_te)

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                              num_workers=NUM_WORKERS, drop_last=False)
    va_loader = DataLoader(va_ds, batch_size=BATCH_SIZE * 2, shuffle=False, num_workers=NUM_WORKERS)
    dev_loader = DataLoader(dev_ds, batch_size=BATCH_SIZE * 2, shuffle=False, num_workers=NUM_WORKERS)
    test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE * 2, shuffle=False, num_workers=NUM_WORKERS)

    model = PhoBERTMeta(MODEL_NAME, dense_train.shape[1]).to(DEVICE)
    if GRAD_CHECKPOINT and hasattr(model.encoder, "gradient_checkpointing_enable"):
        model.encoder.gradient_checkpointing_enable()

    optimizer = AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    total_steps = EPOCHS * math.ceil(len(train_ds) / BATCH_SIZE)
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=int(WARMUP_RATIO * total_steps),
        num_training_steps=total_steps,
    )
    loss_fn = nn.CrossEntropyLoss(
        weight=class_weights(y_train[tr_idx], CLASS_WEIGHT_MODE).to(DEVICE)
    )
    amp_scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)

    best_f1 = -1.0
    best_state = None
    stale = 0

    for epoch in range(EPOCHS):
        model.train()
        epoch_loss = 0.0
        for batch in train_loader:
            input_ids = batch["input_ids"].to(DEVICE)
            attention_mask = batch["attention_mask"].to(DEVICE)
            dense = batch["dense"].to(DEVICE)
            labels = batch["labels"].to(DEVICE)

            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type=DEVICE, enabled=USE_AMP):
                logits = model(input_ids, attention_mask, dense)
                loss = loss_fn(logits, labels)

            amp_scaler.scale(loss).backward()
            amp_scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            amp_scaler.step(optimizer)
            amp_scaler.update()
            scheduler.step()
            epoch_loss += float(loss.item())

        p_dev = predict(model, dev_loader)
        dev_f1 = f1_score(y_dev, p_dev.argmax(axis=1), average="macro")
        dev_acc = accuracy_score(y_dev, p_dev.argmax(axis=1))
        print(f"  fold {fold} | epoch {epoch + 1}/{EPOCHS} | "
              f"loss={epoch_loss / max(len(train_loader), 1):.4f} | "
              f"dev mf1={dev_f1:.4f} acc={dev_acc:.4f}")

        if dev_f1 > best_f1 + 1e-6:
            best_f1 = dev_f1
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            stale = 0
        else:
            stale += 1
            if stale >= PATIENCE:
                print("  early stopping")
                break

        if DEVICE == "cuda":
            torch.cuda.empty_cache()

    if best_state is not None:
        model.load_state_dict(best_state)

    p_va = predict(model, va_loader)
    p_dev = predict(model, dev_loader)
    p_test = predict(model, test_loader)

    va_f1 = f1_score(y_train[va_idx], p_va.argmax(axis=1), average="macro")
    print(f"  fold {fold} xong | val mf1={va_f1:.4f} | "
          f"best dev mf1={best_f1:.4f} | {time.time() - t0:.0f}s")

    del model, optimizer, scheduler, train_loader, va_loader, dev_loader, test_loader
    if DEVICE == "cuda":
        torch.cuda.empty_cache()

    return p_va, p_dev, p_test, best_f1

## 6. Chạy OOF 3 fold

Mỗi fold train một model riêng; lưu xác suất validation vào OOF, cộng dồn dev/test rồi chia cho số fold.

In [6]:
# ============================================================
# 6. OOF LOOP
# ============================================================
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
splits = list(skf.split(np.zeros(len(y_train)), y_train))

oof = np.zeros((len(y_train), 4), dtype=np.float32)
dev_sum = np.zeros((len(y_dev), 4), dtype=np.float32)
test_sum = np.zeros((len(texts_test), 4), dtype=np.float32)

t_all = time.time()
fold_scores = []

for fold, (tr_idx, va_idx) in enumerate(splits):
    print(f"\n===== FOLD {fold} =====")
    p_va, p_dev, p_test, best_dev_f1 = run_fold(fold, tr_idx, va_idx)
    oof[va_idx] = p_va
    dev_sum += p_dev / N_FOLDS
    test_sum += p_test / N_FOLDS
    fold_scores.append(best_dev_f1)

oof_pred = oof.argmax(axis=1)
oof_mf1 = f1_score(y_train, oof_pred, average="macro")
oof_acc = accuracy_score(y_train, oof_pred)
print(f"\nOOF macro-F1 = {oof_mf1:.4f} | accuracy = {oof_acc:.4f}")

np.save(OOF_DIR / "phobert_oof.npy", oof)
np.save(OOF_DIR / "phobert_dev.npy", dev_sum)
np.save(OOF_DIR / "phobert_test.npy", test_sum)
print("Đã lưu xác suất vào:", OOF_DIR)
print("Tổng thời gian:", f"{(time.time() - t_all) / 60:.1f} phút")


===== FOLD 0 =====


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 12280.83it/s]
[transformers] RobertaModel LOAD REPORT from: vinai/phobert-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.decoder.bias      | UNEXPECTED |  | 
lm_head.decoder.weight    | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
[transformers] `use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


  fold 0 | epoch 1/3 | loss=0.6944 | dev mf1=0.6441 acc=0.8881
  fold 0 | epoch 2/3 | loss=0.4503 | dev mf1=0.6749 acc=0.8855
  fold 0 | epoch 3/3 | loss=0.3311 | dev mf1=0.7072 acc=0.9006
  fold 0 xong | val mf1=0.7589 | best dev mf1=0.7072 | 1257s

===== FOLD 1 =====


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 7106.75it/s]
[transformers] RobertaModel LOAD REPORT from: vinai/phobert-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.decoder.bias      | UNEXPECTED |  | 
lm_head.decoder.weight    | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  fold 1 | epoch 1/3 | loss=0.6928 | dev mf1=0.6555 acc=0.8855
  fold 1 | epoch 2/3 | loss=0.4521 | dev mf1=0.6646 acc=0.8855
  fold 1 | epoch 3/3 | loss=0.3240 | dev mf1=0.6813 acc=0.8931
  fold 1 xong | val mf1=0.7699 | best dev mf1=0.6813 | 1071s

===== FOLD 2 =====


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 12442.48it/s]
[transformers] RobertaModel LOAD REPORT from: vinai/phobert-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.decoder.bias      | UNEXPECTED |  | 
lm_head.decoder.weight    | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  fold 2 | epoch 1/3 | loss=0.7272 | dev mf1=0.6131 acc=0.8623
  fold 2 | epoch 2/3 | loss=0.4761 | dev mf1=0.6634 acc=0.8899
  fold 2 | epoch 3/3 | loss=0.3601 | dev mf1=0.6967 acc=0.8912
  fold 2 xong | val mf1=0.7341 | best dev mf1=0.6967 | 1151s

OOF macro-F1 = 0.7544 | accuracy = 0.9008
Đã lưu xác suất vào: D:\DS102_NguyenKhacHuy_24520679\Final project\artifacts\probs\oof_phobert
Tổng thời gian: 58.0 phút


## 7. Tổng hợp

Sau khi chạy xong, mở `src/stacking.ipynb` — notebook sẽ tự phát hiện `artifacts/probs/oof_phobert/` và thêm PhoBERT làm base model thứ 6.

**Nếu OOM:** giảm `BATCH_SIZE = 4`, giữ `GRAD_CHECKPOINT=True`, hoặc giảm `MAX_LEN = 96`.
**Nếu dev macro-F1 thấp bất thường:** kiểm tra `comment_seg` (cell 4 của `phobert_preprocess.ipynb`) và thử `CLASS_WEIGHT_MODE="balanced"`.